# Hybrid KD Mobile Deployment Entry

Notebook khóa quyết định sản phẩm đưa Hybrid KD lên mobile mà **không thay đổi**
kết luận khoa học: Sequence KD vẫn là phương pháp chính, Hybrid vẫn là ablation.

## Input bắt buộc — đúng 4 output

1. `HYB_42`;
2. `HYB_123`;
3. `HYB_2026`;
4. `HYB_Closeout`.

Runtime: `CPU / Accelerator None / Internet Off`.

In [1]:
import csv
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT", "/kaggle/working/HYB_Mobile_Entry"
))
OUT.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

def exactly_one(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (name, matches)
    return matches[0]

def verify_seed(marker_path, expected_seed):
    marker = read_json(marker_path)
    assert marker["status"] == "complete"
    assert marker["experiment_id"] == "EXPANSIONNET_S1_HYBRID_KD"
    assert int(marker["seed"]) == expected_seed
    assert marker["selection_split"] == "validation"
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
    manifest_ref = marker["manifest"]["csv"]
    manifest_path = marker_path.parent / manifest_ref["relative_path"]
    assert sha256(manifest_path) == manifest_ref["sha256"]
    with manifest_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(manifest_ref["rows"])
    for row in rows:
        artifact = marker_path.parent / row["relative_path"]
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    checkpoint_ref = marker["best_checkpoint"]
    checkpoint = marker_path.parent / checkpoint_ref["relative_path"]
    assert checkpoint.is_file()
    assert sha256(checkpoint) == checkpoint_ref["sha256"]
    cider = float(marker["best_result"]["metrics"]["CIDEr-D"])
    return marker, checkpoint, cider, len(rows)

closeout_path = exactly_one("HYB_FINAL_LOCK.json")
closeout = read_json(closeout_path)
assert closeout["status"] == "closed"
assert closeout["role_in_report"] == "ablation"
assert closeout["android_deployment_authorized_for_hybrid"] is False

records = []
for seed in (42, 123, 2026):
    marker_path = exactly_one(f"HYB_{seed}_COMPLETE.json")
    marker, checkpoint, cider, manifest_rows = verify_seed(marker_path, seed)
    records.append({
        "seed": seed,
        "validation_cider_d": cider,
        "checkpoint_name": checkpoint.name,
        "checkpoint_sha256": sha256(checkpoint),
        "checkpoint_bytes": checkpoint.stat().st_size,
        "source_marker_name": marker_path.name,
        "source_marker_sha256": sha256(marker_path),
        "manifest_rows_verified": manifest_rows,
    })

ranked = sorted(records, key=lambda row: (-row["validation_cider_d"], row["seed"]))
selected = ranked[0]
assert selected["seed"] == 2026, ranked

contract = {
    "schema_version": "hyb_mobile_entry_v1",
    "created_at_utc": now(),
    "status": "authorized_by_user",
    "experiment_id": "HYB_Mobile_Entry",
    "method": "Hybrid Sequence and Dense-Mapped Logit Knowledge Distillation",
    "student": "S1 MobileNetV3-Large",
    "scientific_role": "ablation",
    "scientific_primary_method_unchanged": "Sequence KD T1->S1",
    "product_decision_override": True,
    "override_scope": "mobile deployment and product comparison only",
    "selection_split": "validation",
    "selection_metric": "CIDEr-D",
    "ranked_candidates": ranked,
    "selected": selected,
    "parallel_next": [
        "HYB_ONNX_FP32",
        "HYB_Calib_Lock",
    ],
    "test_set_status": "sealed",
    "test_set_used": False,
}
contract_path = OUT / "HYB_MOBILE_ENTRY_CONTRACT.json"
write_json(contract_path, contract)

marker = {
    "schema_version": "hyb_mobile_entry_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "selected_seed": selected["seed"],
    "selected_validation_cider_d": selected["validation_cider_d"],
    "selected_checkpoint": selected,
    "contract": {
        "relative_path": contract_path.name,
        "sha256": sha256(contract_path),
    },
    "product_override": True,
    "scientific_conclusion_changed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(OUT / "HYB_MOBILE_ENTRY_COMPLETE.json", marker)

print("Mobile entry             : COMPLETE")
print("Selected seed            :", selected["seed"])
print("Validation CIDEr-D x100  :", round(selected["validation_cider_d"] * 100, 2))
print("Scientific role          : ABLATION (unchanged)")
print("Product deployment       : USER AUTHORIZED")
print("Next in parallel         : ONNX FP32 + Calibration Lock")

Mobile entry             : COMPLETE
Selected seed            : 2026
Validation CIDEr-D x100  : 110.07
Scientific role          : ABLATION (unchanged)
Product deployment       : USER AUTHORIZED
Next in parallel         : ONNX FP32 + Calibration Lock
